# Weight Alignment Analysis

Analyze how weight matrices align: QK vs OV circuits, cross-head similarity,
cross-layer similarity, embed-unembed alignment, and MLP weight alignment.

## Why This Matters

Weight alignment examines the structure and statistics of model parameters. The structure of weights constrains what computations a component can perform, and spectral analysis can reveal functional specialization, redundancy, and low-rank structure.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_alignment_analysis import (
    qk_ov_alignment, cross_head_weight_alignment,
    cross_layer_weight_alignment, embed_weight_alignment,
    mlp_weight_alignment,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## QK-OV Alignment

Does a head look at what it copies? High cosine = aligned QK and OV circuits.

In [ ]:
for layer in range(4):
    print(f'Layer {layer}:')
    for head in range(4):
        result = qk_ov_alignment(model, layer=layer, head=head)
        aligned = '✓' if result['is_aligned'] else '✗'
        print(f"  Head {head}: cos={result['qk_ov_cosine']:+.4f}, "
              f"QK_spec={result['qk_spectral_norm']:.4f}, "
              f"OV_spec={result['ov_spectral_norm']:.4f} [{aligned}]")
    print()

## Cross-Head Weight Alignment

Do heads within a layer do similar things? Low similarity = diverse specialization.

In [ ]:
for layer in range(4):
    result = cross_head_weight_alignment(model, layer=layer)
    diverse = 'diverse' if result['is_diverse'] else 'redundant'
    print(f"Layer {layer}: mean_abs_sim={result['mean_abs_similarity']:.4f} [{diverse}]")
    for p in result['pairs'][:3]:
        print(f"  Head {p['head_a']}↔{p['head_b']}: {p['ov_similarity']:+.4f}")
    print()

## Cross-Layer Weight Alignment

Do layers compute similar transformations, or do they specialize?

In [ ]:
for component in ['ov', 'qk']:
    result = cross_layer_weight_alignment(model, component=component)
    print(f'{component.upper()} circuit cross-layer alignment:')
    print(f"  Mean similarity: {result['mean_similarity']:.4f}")
    for p in result['pairs']:
        print(f"  Layer {p['layer_a']}↔{p['layer_b']}: {p['similarity']:+.4f}")
    print()

## Embed-Unembed Alignment

Are embedding and unembedding matrices aligned? Tied weights → perfect alignment.

In [ ]:
result = embed_weight_alignment(model)
tied = 'TIED' if result['is_tied'] else 'untied'
print(f"Mean token alignment: {result['mean_token_alignment']:.4f} ± {result['std_token_alignment']:.4f} [{tied}]")
print(f"Embed spectral norm: {result['embed_spectral_norm']:.4f}")
print(f"Unembed spectral norm: {result['unembed_spectral_norm']:.4f}")

## MLP Weight Alignment

Are MLP transformations similar across layers?

In [ ]:
result = mlp_weight_alignment(model)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: W_in_sim={p['mean_in_similarity']:+.4f}, "
          f"W_out_sim={p['mean_out_similarity']:+.4f}")